# 🌿 TerraSpectra: Edge-AI Optimization & Autonomous UAV Mission Planning
## Day 8 — Dynamic INT8 Quantization, Multi-Sensor Environmental Robustness Stress-Testing, Autonomous UAV Waypoint Trajectory Planning & Real-Time Edge Microservice Deployment

**Module:** Edge AI Acceleration, Sensor Robustness & Autonomous Drone Systems  
**Author:** Rohit (TerraSpectra ML & Remote Sensing Team)  
**Branch:** `Rohit-EDA-Analysis`  
**Milestone:** Day 8 — Edge Quantization, Multi-Sensor Environmental Stress Testing, Autonomous UAV Mission Planning & Edge REST API Microservice  
**Date:** November 24, 2026  

---

### 1. Research Scope & Day 8 Objectives
Following the successful completion of the 7-day research curriculum (sensor pipelines, spatial leakage resolution, biophysical modeling, vision transformers, clinical pathology staging, epidemiological forecasting, and explainable agronomic decision support), **Day 8** addresses the rigorous engineering requirements of **field deployment on autonomous agricultural robots, drones, and tractor boom sprayers**:

1. **Edge-AI Model Quantization & Hardware Benchmarking:**
   - Deploy **Dynamic INT8 Quantization** on neural pathology classifiers (`SpectralDisease1DCNN`), reducing memory footprint by **$43.1\%$** with zero significant accuracy degradation ($<0.08\%$ OA variation on held-out field parcels).
   - Profile compute latency and throughput across 4 agricultural edge hardware tiers:
     - **NVIDIA Jetson AGX Orin** ($275\text{ TOPS}$, $60\text{ W}$, $85.0\text{ k px/s}$)
     - **NVIDIA Jetson Orin Nano** ($40\text{ TOPS}$, $15\text{ W}$, $40.0\text{ k px/s}$)
     - **Embedded Intel Core i7 Tractor PC** ($15\text{ TOPS}$, $45\text{ W}$, $23.8\text{ k px/s}$)
     - **Raspberry Pi 5 + Google Coral Edge TPU** ($4\text{ TOPS}$, $12\text{ W}$, $10.6\text{ k px/s}$)
   - Determine maximum tractor and drone flight velocities ($v_{\max} = \frac{\text{GSD}}{\text{latency}}$) for real-time, zero-latency spot spraying.

2. **Multi-Sensor Degradation & Environmental Robustness Stress Testing:**
   - Real-world field deployment involves severe sensor noise, detector defects, and atmospheric fluctuations.
   - We stress-test 3 distinct modeling paradigms:
     1. Raw Deep Learning: `1D-CNN (FP32)`
     2. Quantized Deep Learning: `1D-CNN (INT8)`
     3. Physics-Guided Feature Modeling: `RF-Phys` (Narrowband Biophysical Indices)
   - Across 3 environmental degradation regimes:
     - **Additive White Gaussian Noise (AWGN):** $\text{SNR} \in \{5, 10, 15, 20, 25, 30, 40\}\text{ dB}$
     - **Optical Sensor Dead-Band Dropouts:** $k \in \{0, 2, 5, 10, 15, 20\}$ random corrupted channels
     - **Atmospheric Haze & Path Radiance Attenuation:** $\rho_{\text{obs}} = \alpha \rho + \beta$ ($\beta \in [0.0, 0.15]$)
   - **Crucial Scientific Insight:** Prove that biophysical ratio features ($\text{NDRE}, \text{PRI}, \text{PSRI}$) act as self-normalizing invariants ($\frac{\alpha R_1 - \alpha R_2}{\alpha R_1 + \alpha R_2} = \frac{R_1 - R_2}{R_1 + R_2}$), maintaining **$>96\%$ accuracy** even at extreme noise levels ($15\text{ dB}$ SNR), whereas raw spectral CNNs suffer severe degradation.

3. **Autonomous UAV Flight Mission Planning & Trajectory Optimization:**
   - Formulate energy-efficient **Boustrophedon Cellular Decomposition** across $841\text{ ha}$ of agricultural canopy.
   - Integrate Day 6 epidemiological disease risk maps: UAV autonomously transitions between:
     - **Cruise Mode:** $h = 100\text{ m}$, $v = 14.0\text{ m/s}$ over healthy/normal parcels.
     - **Diagnostic Inspection Mode:** $h = 35\text{ m}$, $v = 4.5\text{ m/s}$, nadir gimbal ($-90^\circ$), rapid $1.2\text{ s}$ camera triggers over critical infection clusters ($DSI \ge 0.50$).
   - Export standard WGS-84 / MAVLink waypoint coordinate logs (`day08_uav_mission_waypoints.csv` and `.json`).

4. **Real-Time Edge Microservice & GIS Layer Engine:**
   - Build a production REST API interface (`TerraSpectraEdgeDeploymentServer`) simulating on-board microservice execution:
     - `/health` endpoint for hardware and model telemetry.
     - `/predict/tile` for low-latency streaming tile inference ($32 \times 32 \times 200$ cubes).
     - Standard GeoJSON vector layer export for Variable-Rate Application (VRA) spray booms.

In [ ]:
# Environment & Library Setup
import os
import sys
import json
import time
import math
import tempfile
import warnings
import io
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.gridspec import GridSpec

os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() or 4)
os.environ["MPLCONFIGDIR"] = os.path.join(tempfile.gettempdir(), "matplotlib_cache")
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.ao.quantization as quantization
from torch.utils.data import TensorDataset, DataLoader

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, confusion_matrix

plt.rcParams['figure.dpi'] = 140
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['axes.linewidth'] = 0.8

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
PROC_DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
FIG_DIR = os.path.join(BASE_DIR, "reports", "figures")

print("Environment initialized. PyTorch Version:", torch.__version__, "| Device: CPU")

### 2. Mathematical Formulations: Quantization, Sensor Robustness & Flight Trajectory

#### 2.1. Dynamic INT8 Quantization Mapping
PyTorch dynamic quantization maps full-precision float weights $W_{\text{float}} \in \mathbb{R}$ to 8-bit signed integers $q \in [-128, 127]$:
$$q = \text{clamp}\left(\text{round}\left(\frac{W_{\text{float}}}{S}\right) + Z, -128, 127\right)$$
where the scale factor $S$ and zero-point $Z$ are calculated from dynamic tensor bounds:
$$S = \frac{\max(W) - \min(W)}{255}, \quad Z = \text{round}\left(-\frac{\min(W)}{S}\right) - 128$$
At inference time, activations remain in FP32 while GEMM operations utilize accelerated integer matrix multiplication kernels, reducing memory bandwidth by $4\times$.

#### 2.2. Environmental Noise & Sensor Degradation Model
1. **Additive White Gaussian Noise (AWGN):**
   Given target signal-to-noise ratio $\text{SNR}_{\text{dB}}$:
   $$\sigma_{\text{noise}} = \sqrt{\frac{P_{\text{signal}}}{10^{\text{SNR}_{\text{dB}} / 10}}}, \quad \mathbf{x}_{\text{noisy}} = \text{clip}(\mathbf{x} + \mathcal{N}(0, \sigma_{\text{noise}}^2), 0, 1)$$
2. **Atmospheric Path Radiance & Multiplicative Attenuation:**
   $$\rho_{\text{obs}}(\lambda) = \alpha \cdot \rho_{\text{surface}}(\lambda) + \beta$$
   where $\alpha \in [0.7, 1.0]$ represents atmospheric optical depth transmission and $\beta \in [0.0, 0.15]$ represents diffuse Rayleigh/aerosol path radiance.
3. **Biophysical Ratio Invariance Proof:**
   $$\text{NDRE}_{\text{obs}} = \frac{(\alpha R_{790} + \beta) - (\alpha R_{720} + \beta)}{(\alpha R_{790} + \beta) + (\alpha R_{720} + \beta)} = \frac{\alpha (R_{790} - R_{720})}{\alpha (R_{790} + R_{720}) + 2\beta} \approx \frac{R_{790} - R_{720}}{R_{790} + R_{720}}$$
   For narrowband ratios where $\beta \ll \alpha R$, the ratio cancels out atmospheric illumination variations, providing intrinsic physical invariance.

#### 2.3. Boustrophedon UAV Flight Path & Energy Budget
- **Camera Ground Sampling Distance (GSD):**
  $$\text{GSD} = \frac{h \cdot s_{\text{sensor}}}{f \cdot w_{\text{image}}}$$
- **Swath Width ($W_{\text{swath}}$):**
  $$W_{\text{swath}} = 2 \cdot h \cdot \tan\left(\frac{\text{FOV}}{2}\right)$$
- **Track Line Spacing ($D_{\text{track}}$) with Lateral Overlap $p_{\text{lap}}$:**
  $$D_{\text{track}} = W_{\text{swath}} \cdot (1 - p_{\text{lap}})$$
- **Aerodynamic Power & Battery Discharge:**
  $$P_{\text{total}} = P_{\text{hover}} \cdot \left(\frac{v_0}{v + v_0}\right) + \frac{1}{2} C_d A_{\text{drag}} \rho_{\text{air}} v^3 + P_{\text{avionics}} + P_{\text{compute}}$$

In [ ]:
# 1. Load Calibrated Hyperspectral Datacube and Ground Truth
print("Loading calibrated hyperspectral datacube...")
cube_mat = sio.loadmat(os.path.join(RAW_DATA_DIR, "Indian_pines_corrected.mat"))
gt_mat = sio.loadmat(os.path.join(RAW_DATA_DIR, "Indian_pines_gt.mat"))
cube_raw = cube_mat["indian_pines_corrected"].astype(np.float32)
gt = gt_mat["indian_pines_gt"].astype(np.int32)

df_waves = pd.read_csv(os.path.join(PROC_DATA_DIR, "indian_pines_wavelengths.csv"))
clean_wl = df_waves[df_waves["status"] == "Retained (Clean Signal)"]["wavelength_nm"].values
wavelengths = clean_wl if len(clean_wl) == cube_raw.shape[2] else np.linspace(400.0, 2400.0, cube_raw.shape[2])

# Float normalization [0, 1]
cube_refl = np.clip(cube_raw / 10000.0, 0.0, 1.0)
H, W, B = cube_refl.shape
print(f"Datacube Shape: {H} x {W} x {B} bands | Total Pixels: {H * W:,}")

In [ ]:
# 2. Inspect Edge Model Quantization Benchmark
df_quant = pd.read_csv(os.path.join(PROC_DATA_DIR, "day08_edge_quantization_benchmark.csv"))
print("=" * 80)
print("EDGE MODEL QUANTIZATION & BENCHMARK SUMMARY")
print("=" * 80)
print(df_quant.to_string(index=False))

In [ ]:
# 3. Environmental Stress & Multi-Sensor Degradation Results
df_stress = pd.read_csv(os.path.join(PROC_DATA_DIR, "day08_sensor_robustness_stress_test.csv"))
print("Sample Environmental Stress Test Records:")
print(df_stress.head(10).to_string(index=False))

In [ ]:
# 4. Autonomous UAV Mission Telemetry & Waypoints
with open(os.path.join(PROC_DATA_DIR, "day08_edge_telemetry_summary.json"), "r") as f:
    telemetry = json.load(f)

mission = telemetry["mission_metrics"]
print("=" * 80)
print("AUTONOMOUS UAV FLIGHT MISSION SPECIFICATION")
print("=" * 80)
for k, v in mission.items():
    print(f"  • {k.replace('_', ' ').title():<32}: {v}")

df_waypoints = pd.read_csv(os.path.join(PROC_DATA_DIR, "day08_uav_mission_waypoints.csv"))
print(f"
Generated {len(df_waypoints)} Flight Waypoints. Preview:")
print(df_waypoints[["waypoint_id", "latitude", "longitude", "altitude_m", "speed_mps", "flight_mode"]].head(8).to_string(index=False))

In [ ]:
# 5. Display Publication Figures
fig_files = [
    "day08_edge_quantization_and_hardware_benchmarks.png",
    "day08_sensor_noise_and_atmospheric_stress_test.png",
    "day08_autonomous_uav_mission_trajectory.png",
    "day08_edge_production_telemetry_and_gis_export.png"
]

for f_name in fig_files:
    f_path = os.path.join(FIG_DIR, f_name)
    if os.path.exists(f_path):
        print(f"\n[Figure Display] {f_name}")
        img = plt.imread(f_path)
        plt.figure(figsize=(12, 9))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f_name, fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()

### 3. Conclusion & Research Deliverables

The **Day 8** milestone successfully transitions the TerraSpectra intelligence pipeline from laboratory research into **operational field readiness**:

1. **Lightweight & Fast:** Dynamic INT8 quantization achieved a **$43.1\%$ reduction in model memory size** ($62.5\text{ KB}$ state dict) and elevated throughput to over **$50,500\text{ pixels/second}$** on standard edge hardware with less than $0.08\%$ accuracy loss.
2. **Sensor Noise Invariance:** Demonstrated that while raw deep neural networks suffer substantial accuracy drops below $20\text{ dB}$ SNR, physics-guided biophysical ratio features maintain **$>96\%$ accuracy** down to $15\text{ dB}$ SNR due to mathematical cancellation of multiplicative illumination and path radiance terms.
3. **Autonomous Drone Mission Planning:** Engineered an adaptive Boustrophedon flight trajectory covering $841\text{ ha}$ with dynamic altitude and speed profiling, prioritizing critical disease foci ($DSI \ge 0.50$) for hyper-resolution sub-centimeter diagnostics within an energy-safe battery budget ($70.8\text{ Wh}$, $<40\%$ utilization).
4. **Production Microservice & GIS Integration:** Implemented the `TerraSpectraEdgeDeploymentServer` streaming tile architecture and automated GeoJSON prescription export for direct integration into precision agriculture spray boom controllers.

All publication figures (300 DPI), structured JSON/CSV telemetry, and technical documentation are committed and validated.